# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****2126
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
# Fetch the schema dynamically so the LLM knows the table/column names
schema = db.get_table_info()

SYSTEM_PROMPT = f"""You are a careful SQLite analyst.

Here is the database schema:
{schema}

Rules:
- Think step-by-step. 
- You often need to JOIN 'Customer' and 'Invoice' tables to find data by name.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
"""

In [6]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:14b", temperature=0)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [8]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (a281d980-3ced-4feb-9803-3a5d89ffd5aa)
 Call ID: a281d980-3ced-4feb-9803-3a5d89ffd5aa
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(16,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fd5aea25-6b0e-4ba4-84ac-8a5bb709e782)
 Call ID: fd5aea25-6b0e-4ba4-84ac-8a5bb709e782
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(5.94,)]
================================== Ai Message ===

In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

The question "What were the titles?" is ambiguous because multiple tables contain 'Title' columns (e.g., `Album`, `Track`). Could you clarify which table's titles you're referring to? For example:
- Are you asking for **album titles**?
- Or **track titles**?
- Something else?

Let me know so I can provide the correct information.


## Add memory

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

In [8]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [12]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (139dbfd9-7b43-4782-ae07-62f45cbed504)
 Call ID: 139dbfd9-7b43-4782-ae07-62f45cbed504
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(16,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (de7ced56-b3c6-4c4f-837e-70f8fd3336e1)
 Call ID: de7ced56-b3c6-4c4f-837e-70f8fd3336e1
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(5.94,)]
================================== Ai Message ===

In [13]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (82c98ad4-39dd-4859-94bf-ddaa485c6b8e)
 Call ID: 82c98ad4-39dd-4859-94bf-ddaa485c6b8e
  Args:
    query: SELECT Track.Name FROM Track JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId WHERE InvoiceLine.InvoiceId = (SELECT InvoiceId FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1);
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]
================================== Ai Message ==================================

The titles on your last invoice were:  
1. Holier Than Thou  
2. Through The Never  
3. My Friend Of Misery  
4. The Wait  
5. Blitzkrieg  
6. So What  

Let me know if you need 

## Try your own queries
Now that there is memory, check the agents recall!

In [15]:
question = "Show me customers from Brazil"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Show me customers from Brazil
================================== Ai Message ==================================
Tool Calls:
  execute_sql (5d548d13-e847-4baa-b7dd-22a6cd14c491)
 Call ID: 5d548d13-e847-4baa-b7dd-22a6cd14c491
  Args:
    query: SELECT * FROM Customer WHERE Country = 'Brazil' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(1, 'Luís', 'Gonçalves', 'Embraer - Empresa Brasileira de Aeronáutica S.A.', 'Av. Brigadeiro Faria Lima, 2170', 'São José dos Campos', 'SP', 'Brazil', '12227-000', '+55 (12) 3923-5555', '+55 (12) 3923-5566', 'luisg@embraer.com.br', 3), (10, 'Eduardo', 'Martins', 'Woodstock Discos', 'Rua Dr. Falcão Filho, 155', 'São Paulo', 'SP', 'Brazil', '01007-010', '+55 (11) 3033-5446', '+55 (11) 3033-4564', 'eduardo@woodstock.com.br', 4), (11, 'Alexandre', 'Rocha', 'Banco do Brasil S.A.', 'Av. Paulista, 2022', 'São Paulo', 'S

In [ ]:
question = "What are the top 5 genres?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What are the top 5 genres?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (09720e60-f93c-477a-b8cd-4deaad49a1b1)
 Call ID: 09720e60-f93c-477a-b8cd-4deaad49a1b1
  Args:
    query: SELECT Genre.Name, COUNT(*) AS TrackCount FROM Track JOIN Genre ON Track.GenreId = Genre.GenreId GROUP BY Genre.GenreId ORDER BY TrackCount DESC LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Rock', 1297), ('Latin', 579), ('Metal', 374), ('Alternative & Punk', 332), ('Jazz', 130)]


In [9]:
question = "Find invoices over $10"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Find invoices over $10
================================== Ai Message ==================================
Tool Calls:
  execute_sql (a27ebc5e-fd16-444b-ad1f-4fa91de7ed2b)
 Call ID: a27ebc5e-fd16-444b-ad1f-4fa91de7ed2b
  Args:
    query: SELECT * FROM Invoice WHERE Total > 10 LIMIT 5
================================= Tool Message =================================
Name: execute_sql

[(5, 23, '2009-01-11 00:00:00', '69 Salem Street', 'Boston', 'MA', 'USA', '2113', 13.86), (12, 2, '2009-02-11 00:00:00', 'Theodor-Heuss-Straße 34', 'Stuttgart', None, 'Germany', '70174', 13.86), (19, 40, '2009-03-14 00:00:00', '8, Rue Hanovre', 'Paris', None, 'France', '75002', 13.86), (26, 19, '2009-04-14 00:00:00', '1 Infinite Loop', 'Cupertino', 'CA', 'USA', '95014', 13.86), (33, 57, '2009-05-15 00:00:00', 'Calle Lira, 198', 'Santiago', None, 'Chile', None, 13.86)]
================================== Ai Message =================

In [9]:
question = "Show tracks by AC/DC"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},          
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Show tracks by AC/DC
================================== Ai Message ==================================
Tool Calls:
  execute_sql (e48ec5e4-cbb3-4881-a7c4-1226f1872afd)
 Call ID: e48ec5e4-cbb3-4881-a7c4-1226f1872afd
  Args:
    query: SELECT Track.Name FROM Track JOIN Album ON Track.AlbumId = Album.AlbumId JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name = 'AC/DC';
================================= Tool Message =================================
Name: execute_sql

[('For Those About To Rock (We Salute You)',), ('Put The Finger On You',), ("Let's Get It Up",), ('Inject The Venom',), ('Snowballed',), ('Evil Walks',), ('C.O.D.',), ('Breaking The Rules',), ('Night Of The Long Knives',), ('Spellbound',), ('Go Down',), ('Dog Eat Dog',), ('Let There Be Rock',), ('Bad Boy Boogie',), ('Problem Child',), ('Overdose',), ("Hell Ain't A Bad Place To Be",), ('Whole Lotta Rosie',)]
========================

In [10]:
question = "List employees and their managers"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

List employees and their managers
================================== Ai Message ==================================
Tool Calls:
  execute_sql (4157a565-9a91-469c-9e23-a0b043790661)
 Call ID: 4157a565-9a91-469c-9e23-a0b043790661
  Args:
    query: SELECT e.FirstName || ' ' || e.LastName AS Employee, COALESCE(m.FirstName || ' ' || m.LastName, 'No Manager') AS Manager FROM Employee e LEFT JOIN Employee m ON e.ReportsTo = m.EmployeeId;
================================= Tool Message =================================
Name: execute_sql

[('Andrew Adams', 'No Manager'), ('Nancy Edwards', 'Andrew Adams'), ('Jane Peacock', 'Nancy Edwards'), ('Margaret Park', 'Nancy Edwards'), ('Steve Johnson', 'Nancy Edwards'), ('Michael Mitchell', 'Andrew Adams'), ('Robert King', 'Michael Mitchell'), ('Laura Callahan', 'Michael Mitchell')]
================================== Ai Message ==================================

Here's the 